In [1]:
import pennylane as qp
qml = qp
import numpy as np

In [2]:
n_bits=2
dev = qp.device("default.qubit", wires=range(n_bits))

@qp.qnode(dev)
def two_distant_spins(B, time):
    """Circuit for evolving the state of two distant electrons in a magnetic field.
    
    Args:
        B (float): The strength of the field, assumed to point in the z direction.
        time (float): The time we evolve the electron wavefunction for.

    Returns: 
        array[complex]: The quantum state after evolution.
    """
    e = -1.6e-19
    m_e = 9.1e-31
    alpha = e*B/(2*m_e)
    ##################
    # YOUR CODE HERE #
    ##################
    qp.RZ(-2*alpha*time, wires=0)
    qp.RZ(-2*alpha*time, wires=1)
    
    return qp.state()

In [3]:
n_bits=2
dev = qp.device("default.qubit", wires=range(n_bits))

@qp.qnode(dev)
def two_close_spins_X(alpha, beta, time, n):
    """Circuit for evolving state of two electrons with an X coupling.
    
    Args:
        alpha (float): The strength of the field, assumed to point in the z direction.
        beta (float): The strength of the coupling between electrons.
        time (float): The time we evolve the electron wavefunction for.
        n (int): The number of steps in our Trotterization.

    Returns: 
        array[complex]: The quantum state after evolution.
    """
    
    ##################
    # YOUR CODE HERE #
    ##################
    for _ in range(n):
        qp.IsingXX(-2*beta*time/n, wires=[0,1])
        qp.RZ(-2*alpha*time/n, wires=[0])
        qp.RZ(-2*alpha*time/n, wires=[1])
        
    return qp.state()

In [4]:
n_bits=2
dev = qp.device("default.qubit", wires=range(n_bits))

def ham_close_spins(alpha, beta):
    """Creates the Hamiltonian for two close spins.

    Args:
        alpha (float): The coefficient related to the strength of the field, assumed to point in the z direction.
        beta (list[float]): A vector of coupling coefficients [beta_X, beta_Y, beta_Z].

    Returns:
        qp.Hamiltonian: The Hamiltonian of the system.
    """

    ##################
    # YOUR CODE HERE #
    ##################
    coeffs = [-alpha, -beta[0], -beta[1], -beta[2]] # MODIFY THIS
    obs = [qp.PauliZ(0) + qp.PauliZ(1), 
           qp.PauliX(0) @ qp.PauliX(1), 
           qp.PauliY(0) @ qp.PauliY(1), 
           qp.PauliZ(0) @ qp.PauliZ(1)] # MODIFY THIS
    
    return qp.dot(coeffs, obs) # Return the Hamiltonian using qp.dot

In [5]:
n_bits = 2
dev = qp.device("default.qubit", wires = n_bits)

@qp.qnode(dev)
def two_close_spins(alpha, beta, time, n):
    """Circuit for evolving state of two nearby electrons with an arbitrary coupling.
    
    Args:
        alpha (float): The strength of the field, assumed to point in the z direction.
        beta (array[float]): The coupling strengths beta = [beta_X, beta_Y, beta_Z] between electrons.
        time (float): The time we evolve the electron wavefunction for.
        n (int): The number of steps in our Trotterization.

    Returns: 
        array[complex]: The quantum state after evolution.
    """
    ##################
    # YOUR CODE HERE #
    ##################
    H = ham_close_spins(alpha, beta)
    qp.TrotterProduct(-H, time, n)
    
    return qp.state()